# Negative coefficient preprocessing for QUBO solving

This tutorial shows how to handle negative off-diagonal coefficients using:

- bit-flip preprocessing;
- optional zeroing of remaining negative coefficients.

Bit-flip preprocessing keeps the problem equivalent up to a variable transformation.

Zeroing is different: it changes the QUBO, so it must be explicitly requested.

In [48]:
import torch

from qoolqit import AnalogDeviceWithDMM
from pulser_simulation import QutipBackendV2

from qubosolver import ClassicalSolverType, QUBOInstance
from qubosolver.config import (
    BitFlipPreprocessingConfig,
    ClassicalConfig,
    LocalEmulator,
    SolverConfig,
)
from qubosolver.pipeline.bitflip_preprocessing import has_negative_offdiagonal
from qubosolver.solver import QuboSolverClassical, QuboSolverQuantum

In [49]:
def is_trivial_bitstring(bitstring: torch.Tensor) -> bool:
    """Return True if a bitstring is all zeros or all ones."""
    return bool(torch.all(bitstring == 0) or torch.all(bitstring == 1))


def print_solution(name, solution):
    print(name)
    print("Bitstrings:")
    print(solution.bitstrings)
    print("Costs:")
    print(solution.costs)

## 1. A QUBO with negative off-diagonal coefficients

We start with a small QUBO that contains negative off-diagonal coefficients.

The example is chosen so that:

- the solution is not trivial;
- bit-flip preprocessing can remove all negative off-diagonal coefficients.

In [50]:
Q = torch.tensor(
    [
        [1.0, -3.0, -2.0, 1.0],
        [-3.0, 6.0, 3.0, -1.0],
        [-2.0, 3.0, 2.0, -1.0],
        [1.0, -1.0, -1.0, -1.0],
    ]
)

print("QUBO:")
print(Q)

print("\nHas negative off-diagonal coefficients:")
print(has_negative_offdiagonal(Q))

QUBO:
tensor([[ 1., -3., -2.,  1.],
        [-3.,  6.,  3., -1.],
        [-2.,  3.,  2., -1.],
        [ 1., -1., -1., -1.]])

Has negative off-diagonal coefficients:
True


## 2. Bit-flip preprocessing

The bit-flip preprocessing chooses which binary variables should be complemented.

For each variable:

- if `flip_vector[i] = 0`, the variable is kept unchanged;
- if `flip_vector[i] = 1`, the variable is replaced by its complement.

Internally, the flip vector is selected by solving a variant of max-cut problem with GLPK.

The objective is to minimize the total weight of negative off-diagonal coefficients remaining after the flips.

In [51]:
config_bitflip = SolverConfig(
    use_quantum=False,
    do_preprocessing=True,
    activate_trivial_solutions=False,
    bitflip_preprocessing=BitFlipPreprocessingConfig(
        enabled=True,
        time_limit_s=5.0,
    ),
)

solver_bitflip = QuboSolverClassical(QUBOInstance(Q), config_bitflip)
solver_bitflip.preprocess()

print("Bitflip applied:", solver_bitflip.fixtures.bitflip_applied)
print("Bitflip vector:", solver_bitflip.fixtures.bitflip_vector)
print("Bitflip status:", solver_bitflip.fixtures.bitflip_status)
print("Bitflip metrics:", solver_bitflip.fixtures.bitflip_metrics)

print("\nPreprocessed QUBO:")
print(solver_bitflip.instance.coefficients)

print("\nHas negative off-diagonal coefficients after preprocessing:")
print(has_negative_offdiagonal(solver_bitflip.instance.coefficients))

Bitflip applied: True
Bitflip vector: tensor([1, 0, 0, 1])
Bitflip status: OPTIMAL
Bitflip metrics: {'n_edges': 6, 'neg_count_before': 4, 'neg_count_after': 0, 'neg_count_reduction_pct': 100.0, 'neg_weight_before': 7.0, 'neg_weight_after': 0.0, 'neg_weight_reduction_pct': 100.0, 'objective_value': -7.0}

Preprocessed QUBO:
tensor([[-3.,  3.,  2.,  1.],
        [ 3., -2.,  3.,  1.],
        [ 2.,  3., -4.,  1.],
        [ 1.,  1.,  1., -1.]])

Has negative off-diagonal coefficients after preprocessing:
False


In [52]:
flip_vector = solver_bitflip.fixtures.bitflip_vector

print("Flip vector:", flip_vector)

for i, flip in enumerate(flip_vector.tolist()):
    if flip == 0:
        print(f"x_{i} = y_{i}")
    else:
        print(f"x_{i} = 1 - y_{i}")

Flip vector: tensor([1, 0, 0, 1])
x_0 = 1 - y_0
x_1 = y_1
x_2 = y_2
x_3 = 1 - y_3


## 3. Classical sanity check with CPLEX

We solve the same QUBO with CPLEX:

1. without preprocessing;
2. with bit-flip preprocessing.

Since bit-flip preprocessing is an equivalent transformation, the returned solution should have the same objective value in the original QUBO space.

In [53]:
classical_config = ClassicalConfig(
    classical_solver_type=ClassicalSolverType.CPLEX,
    max_bitstrings=1,
)

config_without_preprocessing = SolverConfig(
    use_quantum=False,
    do_preprocessing=False,
    activate_trivial_solutions=False,
    classical=classical_config,
)

solver_without_preprocessing = QuboSolverClassical(
    QUBOInstance(Q),
    config_without_preprocessing,
)

solution_without_preprocessing = solver_without_preprocessing.solve()

print_solution("Solution without preprocessing:", solution_without_preprocessing)

best_bitstring_without_preprocessing = solution_without_preprocessing.bitstrings[0]
print("\nIs trivial solution:", is_trivial_bitstring(best_bitstring_without_preprocessing))

Solution without preprocessing:
Bitstrings:
tensor([[1, 0, 1, 1]], dtype=torch.int32)
Costs:
tensor([-2.])

Is trivial solution: False


In [54]:
config_with_bitflip = SolverConfig(
    use_quantum=False,
    do_preprocessing=True,
    activate_trivial_solutions=False,
    classical=classical_config,
    bitflip_preprocessing=BitFlipPreprocessingConfig(
        enabled=True,
        time_limit_s=5.0,
    ),
)

solver_with_bitflip = QuboSolverClassical(
    QUBOInstance(Q),
    config_with_bitflip,
)

solution_with_bitflip = solver_with_bitflip.solve()

print("Solution with bit-flip preprocessing:")
print("Bitstrings:", solution_with_bitflip.bitstrings)
print("Costs:", solution_with_bitflip.costs)
print("Status:", solution_with_bitflip.solution_status)

Solution with bit-flip preprocessing:
Bitstrings: tensor([[1, 0, 1, 1]], dtype=torch.int32)
Costs: tensor([-2.])
Status: preprocessed


In [55]:
best_bitstring_without_preprocessing = solution_without_preprocessing.bitstrings[0]
best_bitstring_with_bitflip = solution_with_bitflip.bitstrings[0]

best_cost_without_preprocessing = solution_without_preprocessing.costs.min()
best_cost_with_bitflip = solution_with_bitflip.costs.min()

print("Best cost without preprocessing:", float(best_cost_without_preprocessing))
print("Best cost with bit-flip preprocessing:", float(best_cost_with_bitflip))

print(
    "Same best cost:",
    torch.isclose(
        best_cost_without_preprocessing,
        best_cost_with_bitflip,
    ).item(),
)

print("\nIs solution without preprocessing trivial:")
print(is_trivial_bitstring(best_bitstring_without_preprocessing))

print("\nIs solution with bit-flip preprocessing trivial:")
print(is_trivial_bitstring(best_bitstring_with_bitflip))

assert torch.isclose(best_cost_without_preprocessing, best_cost_with_bitflip)
assert not is_trivial_bitstring(best_bitstring_without_preprocessing)
assert not is_trivial_bitstring(best_bitstring_with_bitflip)

Best cost without preprocessing: -2.0
Best cost with bit-flip preprocessing: -2.0
Same best cost: True

Is solution without preprocessing trivial:
False

Is solution with bit-flip preprocessing trivial:
False


## 4. When bit-flip preprocessing is not enough

Bit-flip preprocessing reduces negative off-diagonal coefficients by choosing a good flip vector.

However, it does not always remove all negative coefficients.

In the next example, bit-flip preprocessing reduces the negative weight, but one negative coefficient remains.

In [56]:
Q_hard = torch.tensor(
    [
        [0.0, -2.0, 1.0, 1.0],
        [-2.0, 0.0, -2.0, 1.0],
        [1.0, -2.0, 0.0, -2.0],
        [1.0, 1.0, -2.0, 0.0],
    ]
)

print("QUBO:")
print(Q_hard)

print("\nHas negative off-diagonal coefficients:")
print(has_negative_offdiagonal(Q_hard))

QUBO:
tensor([[ 0., -2.,  1.,  1.],
        [-2.,  0., -2.,  1.],
        [ 1., -2.,  0., -2.],
        [ 1.,  1., -2.,  0.]])

Has negative off-diagonal coefficients:
True


In [57]:
config_error = SolverConfig(
    use_quantum=False,
    do_preprocessing=True,
    activate_trivial_solutions=False,
    negative_handling="error",
    bitflip_preprocessing=BitFlipPreprocessingConfig(
        enabled=True,
        time_limit_s=5.0,
    ),
)

solver_error = QuboSolverClassical(QUBOInstance(Q_hard), config_error)
solver_error.preprocess()

print("Bitflip applied:", solver_error.fixtures.bitflip_applied)
print("Zeroing applied:", solver_error.fixtures.zeroing_applied)
print("Bitflip vector:", solver_error.fixtures.bitflip_vector)
print("Bitflip status:", solver_error.fixtures.bitflip_status)
print("Bitflip metrics:", solver_error.fixtures.bitflip_metrics)

print("\nPreprocessed QUBO:")
print(solver_error.instance.coefficients)

print("\nHas negative off-diagonal coefficients after preprocessing:")
print(has_negative_offdiagonal(solver_error.instance.coefficients))

Bitflip applied: True
Zeroing applied: False
Bitflip vector: tensor([1, 0, 1, 0])
Bitflip status: OPTIMAL
Bitflip metrics: {'n_edges': 6, 'neg_count_before': 3, 'neg_count_after': 1, 'neg_count_reduction_pct': 66.66666666666667, 'neg_weight_before': 6.0, 'neg_weight_after': 1.0, 'neg_weight_reduction_pct': 83.33333333333333, 'objective_value': -5.0}

Preprocessed QUBO:
tensor([[-2.,  2.,  1., -1.],
        [ 2., -8.,  2.,  1.],
        [ 1.,  2., -2.,  2.],
        [-1.,  1.,  2., -2.]])

Has negative off-diagonal coefficients after preprocessing:
True


## 5. Explicit zeroing of remaining negative coefficients

If negative off-diagonal coefficients remain after bit-flip preprocessing, the user can explicitly choose to set them to zero.

This is controlled by:

`negative_handling="zeroing"`

This makes the QUBO compatible with the quantum solver, but it changes the QUBO objective. It should therefore be used only when this approximation is acceptable.

In [58]:
config_zeroing = SolverConfig(
    use_quantum=False,
    do_preprocessing=True,
    activate_trivial_solutions=False,
    negative_handling="zeroing",
    bitflip_preprocessing=BitFlipPreprocessingConfig(
        enabled=True,
        time_limit_s=5.0,
    ),
)

solver_zeroing = QuboSolverClassical(QUBOInstance(Q_hard), config_zeroing)
solver_zeroing.preprocess()

print("Bitflip applied:", solver_zeroing.fixtures.bitflip_applied)
print("Zeroing applied:", solver_zeroing.fixtures.zeroing_applied)
print("Bitflip vector:", solver_zeroing.fixtures.bitflip_vector)
print("Bitflip status:", solver_zeroing.fixtures.bitflip_status)
print("Bitflip metrics:", solver_zeroing.fixtures.bitflip_metrics)

print("\nPreprocessed QUBO:")
print(solver_zeroing.instance.coefficients)

print("\nHas negative off-diagonal coefficients after preprocessing:")
print(has_negative_offdiagonal(solver_zeroing.instance.coefficients))

Bitflip applied: True
Zeroing applied: True
Bitflip vector: tensor([1, 0, 1, 0])
Bitflip status: OPTIMAL
Bitflip metrics: {'n_edges': 6, 'neg_count_before': 3, 'neg_count_after': 1, 'neg_count_reduction_pct': 66.66666666666667, 'neg_weight_before': 6.0, 'neg_weight_after': 1.0, 'neg_weight_reduction_pct': 83.33333333333333, 'objective_value': -5.0}

Preprocessed QUBO:
tensor([[-2.,  2.,  1.,  0.],
        [ 2., -8.,  2.,  1.],
        [ 1.,  2., -2.,  2.],
        [ 0.,  1.,  2., -2.]])

Has negative off-diagonal coefficients after preprocessing:
False


## 6. Quantum solver behavior

The quantum solver checks the QUBO after preprocessing.

If negative off-diagonal coefficients remain, it raises an error before running the quantum backend.

We first show this behavior without zeroing.

In [59]:
config_quantum_error = SolverConfig(
    use_quantum=True,
    do_preprocessing=True,
    activate_trivial_solutions=False,
    negative_handling="error",
    backend=LocalEmulator(
        backend_type=QutipBackendV2,
        num_shots=100,
    ),
    device=AnalogDeviceWithDMM(),
    bitflip_preprocessing=BitFlipPreprocessingConfig(
        enabled=True,
        time_limit_s=5.0,
    ),
)

quantum_solver_error = QuboSolverQuantum(QUBOInstance(Q_hard), config_quantum_error)

try:
    quantum_solver_error.solve()
except ValueError as exc:
    print("Error:")
    print(exc)

Error:
Quantum solver does not handle off-diagonal negative coefficients. Preprocessing did not remove all negative coefficients.


Now we explicitly enable zeroing.

The bit-flip preprocessing reduces the negative coefficients first. Then zeroing removes the remaining negative coefficients. The resulting QUBO can be passed to the quantum backend.

In [66]:
config_quantum_zeroing = SolverConfig(
    use_quantum=True,
    do_preprocessing=True,
    activate_trivial_solutions=False,
    negative_handling="zeroing",
    backend=LocalEmulator(
        backend_type=QutipBackendV2,
        num_shots=500,
    ),
    device=AnalogDeviceWithDMM(),
    bitflip_preprocessing=BitFlipPreprocessingConfig(
        enabled=True,
        time_limit_s=300.0,
    ),
)

preprocessed_quantum_solver = QuboSolverQuantum(QUBOInstance(Q_hard), config_quantum_zeroing)
preprocessed_quantum_solver.preprocess()

print("Bitflip applied:", preprocessed_quantum_solver.fixtures.bitflip_applied)
print("Zeroing applied:", preprocessed_quantum_solver.fixtures.zeroing_applied)

print("\nQUBO before quantum solve:")
print(preprocessed_quantum_solver.instance.coefficients)

print("\nHas negative off-diagonal coefficients before quantum solve:")
print(has_negative_offdiagonal(preprocessed_quantum_solver.instance.coefficients))

quantum_solver = QuboSolverQuantum(QUBOInstance(Q_hard), config_quantum_zeroing)
quantum_solution = quantum_solver.solve()

print("\nQuantum solution bitstrings:")
print(quantum_solution.bitstrings)

print("\nQuantum solution costs:")
print(quantum_solution.costs)

Bitflip applied: True
Zeroing applied: True

QUBO before quantum solve:
tensor([[-2.,  2.,  1.,  0.],
        [ 2., -8.,  2.,  1.],
        [ 1.,  2., -2.,  2.],
        [ 0.,  1.,  2., -2.]])

Has negative off-diagonal coefficients before quantum solve:
False

Quantum solution bitstrings:
tensor([[1, 1, 1, 0],
        [1, 1, 1, 1],
        [0, 1, 1, 1],
        [1, 1, 0, 0],
        [0, 0, 1, 1],
        [0, 1, 1, 0]], dtype=torch.int32)

Quantum solution costs:
tensor([-6., -6., -6., -4., -4., -4.])


### Note on the quantum result

The quantum solution returned here is not the optimal solution of the original QUBO.

In this example, the quantum solver runs on a QUBO where the remaining negative off-diagonal coefficients have been set to zero. This makes the problem compatible with the quantum backend, but it also changes the objective optimized by the quantum dynamics.

The returned bitstrings are then evaluated on the original QUBO, which explains why the final costs should be interpreted as heuristic results.

In our ongoing work we are going beyond this approximation by adding a dedicated method to encode negative interactions during the quantum resolution itself, in addition to the bit-flip preprocessing that already reduces them.

## Conclusion

Bit-flip preprocessing provides a practical way to reduce or remove negative off-diagonal QUBO coefficients before quantum solving.

It works by selecting a flip vector with GLPK. The flip vector indicates which binary variables should be complemented, and the transformed QUBO is equivalent to the original problem up to this variable change.

In this tutorial, GLPK is used to compute the flip vector exactly on small QUBOs. This is useful for validation and demonstration, but it is not expected to scale to large industrial instances.

In future versions, this exact GLPK step should be replaced by an efficient classical heuristic for selecting good flip vectors at larger scale.

When bit-flip preprocessing removes all negative off-diagonal coefficients, the QUBO can be passed directly to the quantum solver.

When some negative coefficients remain, `negative_handling="zeroing"` can be enabled explicitly. This makes the QUBO compatible with the quantum backend, but it is an approximation because it changes the QUBO objective.

The current approach is therefore useful as a preprocessing step, but future work should add a dedicated method to represent negative interactions directly during the quantum resolution.